In [ ]:
# imports

from PIL import Image
import matplotlib.pyplot as plt
import os
import numpy as np
import imageio.v2 as imageio
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

In [ ]:
# visualise

# visualise test images from model trained up to epoch 5
# visualising the first 5 images for now

EPOCH = 5
# running from /notebooks so need to cd up a level before into results
base = f"../results/hsi_to_rgb_cyclegan/test_{EPOCH}/images"

fake_dir = os.path.join(base, "fake_B")
real_dir = os.path.join(base, "real_B")

names = sorted(os.listdir(fake_dir))[:5]  # first 5 images

for name in names:
    # .convert("RGB") is needed to avoid colour issues because some PNGs can be paletted or grayscale
    fake = Image.open(os.path.join(fake_dir, name)).convert("RGB")
    real = Image.open(os.path.join(real_dir, name)).convert("RGB")

    # makes it obvious is a matching real image is missing
    if not os.path.exists(os.path.join(real_dir, name)):
        print(f"Missing real_B for {name}, skipping")
        continue

    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    plt.imshow(real)
    plt.title("real_B (ground truth rgb image)")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(fake)
    plt.title("fake_B (generated rgb image)")
    plt.axis("off")

    plt.show()

In [ ]:
# evaluate

# calculates the ssim and psnr values for each of the output images in an experiment, and then averages over these
# ssim = Structural Similarity Index
# psnr = Peak Signal-Noise Ratio


def evaluate_epoch(results_dir, data_range=1.0, channel_axis=2):
    """
    Computes SSIM and PSNR for all fake_B vs real_B image pairs in the results directory for a given CycleGAN/CUT epoch.

    Parameters:
    results_dir (str): path to results/[experiment name]/test_[epoch name]/images directory
    data_range (float): 1.0 for normalised images (I normalised images in calibration)
    channel_axis (int): 2 for RGB images 
    """

    fake_dir = os.path.join(results_dir, "fake_B")
    real_dir = os.path.join(results_dir, "real_B")

    fake_names = sorted(os.listdir(fake_dir))

    # add the ssim and psnr scores for each image to a list
    # will then calculate the mean of this to give experiment results
    ssim_scores = []
    psnr_scores = []

    for i,name in enumerate(fake_names, start=1):
        fake_image_path = os.path.join(fake_dir, name)
        real_image_path = os.path.join(real_dir, name)
        
        # make it obvious if a corresponding real image is missing
        if not os.path.exists(real_image_path):
            print(f"Missing real_B for {name}, skipping")
            continue
    
        # open real and generated/fake images
        fake_image = imageio.imread(fake_image_path).astype(np.float32)
        real_image = imageio.imread(real_image_path).astype(np.float32)

        # check that real and generated output stained rgb images have the same shape
        if fake_image.shape != real_image.shape:
            print("Error: shape mismatch")

        # generate the ssim score for the current image pair and add to list of ssim scores
        ssim_score = ssim(fake_image, real_image, channel_axis=channel_axis, data_range=data_range)
        ssim_scores.append(ssim_score)

        # generate the psnr score for the current image pair and add to list of psnr scores
        psnr_score = psnr(real_image, fake_image, data_range=data_range)
        psnr_scores.append(psnr_score)

        print("SSIM and PSNR scores for output RGB image", i, "computed!")

    # calculate the mean and standard deviations of the ssim and psnr scores to get experiment results
    ssim_mean = np.mean(ssim_scores)
    ssim_std = np.std(ssim_scores)

    psnr_mean = np.mean(psnr_scores)
    psnr_std = np.std(psnr_scores)

    # print the evaluation results for the experiment
    print("Evaluation Results")
    print(f"Images evaluated: {len(ssim_scores)}")
    print(f"SSIM : {ssim_mean:.4f} ± {ssim_std:.4f}")
    # the unit of PSNR is decibels (dB)
    print(f"PSNR : {psnr_mean:.2f} ± {psnr_std:.2f} dB")

# evaluating results from epoch 5
EPOCH = 5
# running from /notebooks so need to cd up a level before into results
results_path = f"../results/hsi_to_rgb_cyclegan/test_{EPOCH}/images"
evaluate_epoch(results_path)


# extension I can add in here to evaluate all epochs in a loop (can do the same with visualisation?)
# then I can plot the table to get a curve of metric value against training epoch, to determine optimal epoch model